### **Column Masking**

In [0]:
%sql
-- privileged readers see the real value; everyone else sees it partially redacted
CREATE OR REPLACE FUNCTION workspace.silver.mask_email(v STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('pii_readers') THEN v
  ELSE regexp_replace(v, '(^.)[^@]*(@.*)$', '$1***$2')   -- p***@example.com
END;

CREATE OR REPLACE FUNCTION workspace.silver.mask_postcode(v STRING)
RETURNS STRING
RETURN CASE
  WHEN is_account_group_member('pii_readers') THEN v
  ELSE concat(split(v, ' ')[0], ' ***')   -- keeps the area code, hides the rest: 'B3 ***'
END;

In [0]:
%sql
ALTER TABLE workspace.silver.customers ALTER COLUMN email    SET MASK workspace.silver.mask_email;
ALTER TABLE workspace.silver.customers ALTER COLUMN postcode SET MASK workspace.silver.mask_postcode;

In [0]:
%sql
SELECT account_id, email, postcode, region FROM workspace.silver.customers LIMIT 5;